# 01 — Sentinel-2 Single Dataset Preprocessing Pipeline
## Automated Satellite Imagery Ingestion, Cloud Removal & GeoTIFF Export

### Overview
This notebook implements an automated, modular, and robust remote-sensing preprocessing pipeline using:
- **PyTorch `Dataset` & `DataLoader`**: Standardized iteration and batch streaming of geospatial target areas.
- **Sentinel Hub API & CDSE OAuth**: Secure satellite retrieval via Copernicus Data Space Ecosystem.
- **eo-learn Cloud & Noise Masking**: SCL-based validity classification, acquisition filtering, and temporal median compositing.
- **Georeferenced GeoTIFF Generation**: Projected UTM output storage with standardized sample hierarchies.
- **Automated Quality Assurance & Visualization**: 3-panel before/mask/after previews and programmatic disk validation.

> **Execution Rule**: Strictly enforced hard cap of `MAX_SAMPLES = 5` to enable verification and testing without full-dataset overhead.


## 1. Environment Setup
We detect whether this notebook is executing in **Google Colab** or a **Local Environment**, configuring mounts and directories dynamically without hardcoding personal system paths.


In [1]:
import os
import sys
from pathlib import Path

# Detect execution platform
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("[INFO] Running in Google Colab environment.")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("[INFO] Google Drive mounted successfully at /content/drive.")
    except Exception as e:
        print(f"[WARN] Drive mount skipped: {e}")
else:
    print("[INFO] Running in Local Environment.")


[INFO] Running in Local Environment.


## 2. Install Dependencies
Installs the required minimal dependencies when running in cloud environments (e.g. Google Colab).
Note: Setuptools is pinned `<81` to maintain compatibility with `pyfilesystem2`.


In [2]:
# Run dependency installation if needed (uncomment for fresh Colab instances)
# !pip install -q eo-learn-core==1.5.7 sentinelhub==3.12.0 "setuptools<81" tifffile rasterio shapely pyproj python-dotenv matplotlib


## 3. Imports
Import required scientific, geospatial, remote sensing, and PyTorch packages.


In [3]:
import csv
import json
import logging
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import tifffile
import torch
from torch.utils.data import DataLoader, Dataset
from pyproj import Transformer
from shapely.geometry import box, shape

from eolearn.core import EOPatch
from sentinelhub import BBox, CRS, Geometry, get_utm_crs

# Preprocessing modules
from s2prep.config import get_collection, get_config
from s2prep.download import SentinelHubDownloadTask
from s2prep.export import write_geotiff
from s2prep.tasks import (
    ComputeValidMaskTask,
    FilterByValidFractionTask,
    MedianCompositeTask,
    NormalizeTask,
)

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version  : {np.__version__}")
print(f"TiffFile ver   : {tifffile.__version__}")


c:\Users\advik\AppData\Local\Programs\Python\Python313\Lib\site-packages\fs\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)  # type: ignore


PyTorch version: 2.14.0+cpu
NumPy version  : 2.3.5
TiffFile ver   : 2026.9.20


## 4. Configuration
Central pipeline configuration. Paths and credentials are retrieved dynamically from environment variables (`DATA_ROOT`, `OUTPUT_ROOT`, `SENTINEL_HUB_CLIENT_ID`, `SENTINEL_HUB_CLIENT_SECRET`).
**Zero hardcoded credentials or private paths**.


In [4]:
@dataclass
class PipelineConfig:
    data_root: Path = field(
        default_factory=lambda: Path(os.getenv("DATA_ROOT", "sample_datasets/coimbatore_aoi"))
    )
    output_root: Path = field(
        default_factory=lambda: Path(os.getenv("OUTPUT_ROOT", "processed_dataset"))
    )
    dataset_name: str = "coimbatore_aoi"
    
    # Credentials (read securely from environment variables)
    sh_client_id: str = field(
        default_factory=lambda: os.getenv("SENTINEL_HUB_CLIENT_ID", os.getenv("SH_CLIENT_ID", "")).strip()
    )
    sh_client_secret: str = field(
        default_factory=lambda: os.getenv("SENTINEL_HUB_CLIENT_SECRET", os.getenv("SH_CLIENT_SECRET", "")).strip()
    )
    sh_provider: str = field(
        default_factory=lambda: os.getenv("SH_PROVIDER", "cdse").strip().lower()
    )

    bands: Tuple[str, ...] = ("B02", "B03", "B04", "B08")  # B, G, R, NIR (10 m native)
    resolution: int = 10                                     # metres per pixel
    maxcc: float = 0.3                                       # 30% scene cloud cover limit
    min_valid_fraction: float = 0.6                          # 60% valid pixel threshold
    reflectance_max: float = 0.3                             # Normalization upper bound
    max_samples: int = 5                                     # Hard cap for verification
    cache_dir: Optional[Path] = field(default_factory=lambda: Path("data/cache"))
    fallback_eopatch_path: Optional[Path] = field(default_factory=lambda: Path("data/eopatches/tile_0000"))

    def has_live_credentials(self) -> bool:
        return bool(self.sh_client_id and self.sh_client_secret)

config = PipelineConfig()
print(f"Dataset Name   : {config.dataset_name}")
print(f"Data Root      : {config.data_root}")
print(f"Output Root    : {config.output_root}")
print(f"Live Auth Ready: {config.has_live_credentials()}")


Dataset Name   : coimbatore_aoi
Data Root      : sample_datasets\coimbatore_aoi
Output Root    : processed_dataset
Live Auth Ready: True


## 5. Dataset Structure
Inspect the input dataset layout. The dataset contains sample coordinates and metadata in `samples.csv` and individual GeoJSON tiles in `tiles/`.


In [5]:
csv_path = config.data_root / "samples.csv"
if csv_path.exists():
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        sample_rows = list(reader)
    print(f"Total samples defined in index: {len(sample_rows)}")
    for r in sample_rows[:3]:
        print(f"  Sample ID: {r['sample_id']} | Name: {r['name']} | Coords: [{r['min_lon']}, {r['min_lat']}, {r['max_lon']}, {r['max_lat']}]")
else:
    print(f"[WARN] No samples.csv found at {csv_path}")


Total samples defined in index: 5
  Sample ID: sample_0000 | Name: Coimbatore_Urban_Core | Coords: [76.949907, 11.011053, 76.961694, 11.022547]
  Sample ID: sample_0001 | Name: Coimbatore_North_Agri | Coords: [76.982505, 11.072754, 76.994296, 11.084246]
  Sample ID: sample_0002 | Name: Singanallur_Lake_Zone | Coords: [77.015106, 10.996454, 77.026894, 11.007946]


## 6. Sentinel Hub & CDSE Configuration
Validates OAuth credentials for the Copernicus Data Space Ecosystem (CDSE). If credentials are missing, the pipeline safely activates the cached/offline verification mode without failing.


In [6]:
if config.has_live_credentials():
    os.environ["SH_CLIENT_ID"] = config.sh_client_id
    os.environ["SH_CLIENT_SECRET"] = config.sh_client_secret
    os.environ["SH_PROVIDER"] = config.sh_provider
    try:
        sh_cfg, provider = get_config()
        collection = get_collection(provider)
        print(f"[SUCCESS] CDSE OAuth configured for provider: {provider}")
    except Exception as e:
        print(f"[WARN] Credentials error: {e}. Falling back to cached verification mode.")
else:
    print("[INFO] Running in Offline/Cached Verification Mode (no CDSE credentials set in environment).")


[SUCCESS] CDSE OAuth configured for provider: cdse


## 7. PyTorch Dataset Class (`SentinelDataset`)
Implements `torch.utils.data.Dataset` to ingest the dataset directory and output standard sample dictionaries:
`{sample_id, coordinates, metadata, source_path}`.


In [7]:
class SentinelDataset(Dataset):
    """
    PyTorch Dataset understanding a Sentinel-2 coordinate/tile dataset structure.
    Returns:
        - sample_id: str (unique identifier)
        - coordinates: [min_lon, min_lat, max_lon, max_lat]
        - metadata: dict (time_interval, location name, etc.)
        - source_path: str (source file path)
    """

    def __init__(self, data_root: Path | str, dataset_name: str = "coimbatore_aoi"):
        self.data_root = Path(data_root)
        self.dataset_name = dataset_name
        self.samples: List[Dict[str, Any]] = []
        self._load_dataset()

    def _load_dataset(self) -> None:
        csv_file = self.data_root / "samples.csv"
        tiles_dir = self.data_root / "tiles"

        if csv_file.exists():
            with open(csv_file, "r", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    sample_id = row.get("sample_id", f"sample_{len(self.samples):04d}")
                    coords = [
                        float(row["min_lon"]),
                        float(row["min_lat"]),
                        float(row["max_lon"]),
                        float(row["max_lat"]),
                    ]
                    self.samples.append({
                        "sample_id": sample_id,
                        "coordinates": coords,
                        "metadata": {
                            "name": row.get("name", sample_id),
                            "center_lon": float(row.get("center_lon", (coords[0] + coords[2]) / 2)),
                            "center_lat": float(row.get("center_lat", (coords[1] + coords[3]) / 2)),
                            "time_interval": (row.get("time_start", "2025-11-01"), row.get("time_end", "2026-02-28")),
                            "dataset_name": self.dataset_name,
                        },
                        "source_path": row.get("file_path", str(csv_file)),
                    })
        elif tiles_dir.exists():
            for gf in sorted(tiles_dir.glob("*.geojson")):
                with open(gf, "r", encoding="utf-8") as f:
                    gj = json.load(f)
                geom = shape(gj["geometry"])
                minx, miny, maxx, maxy = geom.bounds
                props = gj.get("properties", {})
                self.samples.append({
                    "sample_id": props.get("sample_id", gf.stem),
                    "coordinates": [minx, miny, maxx, maxy],
                    "metadata": {
                        "name": props.get("name", gf.stem),
                        "center_lon": (minx + maxx) / 2,
                        "center_lat": (miny + maxy) / 2,
                        "time_interval": tuple(props.get("time_interval", ["2025-11-01", "2026-02-28"])),
                        "dataset_name": self.dataset_name,
                    },
                    "source_path": str(gf),
                })
        else:
            raise FileNotFoundError(f"No samples.csv or tiles/ folder found in: {self.data_root}")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        return self.samples[idx]

dataset = SentinelDataset(config.data_root, dataset_name=config.dataset_name)
print(f"Dataset length: {len(dataset)}")
sample_preview = dataset[0]
print(f"Sample 0: ID={sample_preview['sample_id']}, Coords={sample_preview['coordinates']}")


Dataset length: 5
Sample 0: ID=sample_0000, Coords=[76.949907, 11.011053, 76.961694, 11.022547]


## 8. PyTorch DataLoader
Wrap `SentinelDataset` in a PyTorch `DataLoader` with `batch_size=1, shuffle=False, num_workers=0` and a direct passthrough `collate_fn`.


In [8]:
dataloader = DataLoader(
    dataset,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=lambda batch: batch[0],
)
print("DataLoader initialized successfully.")
first_batch = next(iter(dataloader))
print(f"Batch sample_id: {first_batch['sample_id']}")


DataLoader initialized successfully.
Batch sample_id: sample_0000


## 9. Preprocessing Model (`SentinelPreprocessor`)
Encapsulates:
1. Coordinate-based bounding box projection (`WGS84 -> UTM`).
2. Sentinel Hub Process API download / cached retrieval.
3. `eo-learn` cloud masking (`ComputeValidMaskTask` using SCL scene classification).
4. Timestamp filtering (`FilterByValidFractionTask`).
5. Temporal median compositing (`MedianCompositeTask`).
6. Reflectance normalization (`NormalizeTask`).
7. Georeferenced GeoTIFF generation (`write_geotiff`) with standardized hierarchies.
8. 3-panel visualization generation.


In [9]:
from single_dataset_pipeline import SentinelPreprocessor

preprocessor = SentinelPreprocessor(config)
print("SentinelPreprocessor initialized.")


2026-09-27 21:46:09,445 [INFO] CDSE OAuth configured successfully (provider=cdse).
SentinelPreprocessor initialized.


## 10. Pipeline Execution Function
Integrates the PyTorch DataLoader and Preprocessor with strict `MAX_SAMPLES = 5` hard cap, per-sample error handling, and structured progress logging.


In [10]:
from single_dataset_pipeline import run_single_dataset_pipeline, validate_sample_outputs

print("Pipeline runner ready.")


Pipeline runner ready.


## 11. Run 5 Samples
Execute the pipeline on 5 real samples from the dataset.


In [11]:
results = run_single_dataset_pipeline(config)


2026-09-27 21:46:09,472 [INFO] ======================================================================
2026-09-27 21:46:09,473 [INFO] STARTING SINGLE DATASET PREPROCESSING PIPELINE (Deliverable 1)
2026-09-27 21:46:09,473 [INFO] Target Dataset : coimbatore_aoi
2026-09-27 21:46:09,475 [INFO] Data Root      : C:\Users\advik\OneDrive\Desktop\ADVIK FILES\Skills\coding\projects\Ours\SIH\v3\sample_datasets\coimbatore_aoi
2026-09-27 21:46:09,477 [INFO] Output Root    : C:\Users\advik\OneDrive\Desktop\ADVIK FILES\Skills\coding\projects\Ours\SIH\v3\processed_dataset
2026-09-27 21:46:09,477 [INFO] Hard Cap       : MAX_SAMPLES = 5
2026-09-27 21:46:09,478 [INFO] Live CDSE Auth : ACTIVE
2026-09-27 21:46:09,478 [INFO] ======================================================================
2026-09-27 21:46:09,479 [INFO] Loaded 5 sample(s) from sample_datasets\coimbatore_aoi
2026-09-27 21:46:09,482 [INFO] CDSE OAuth configured successfully (provider=cdse).
2026-09-27 21:46:09,484 [INFO] [1/5] Processing 

## 12. Visualization
Display the generated 3-panel previews (`visualization.png`) directly inside the notebook.
Each preview contains:
1. **BEFORE**: Raw satellite acquisition on the cloudiest date with valid pixel percentage.
2. **CLOUD MASK**: Detected clouds, shadows, and invalid pixels overlaid in red.
3. **AFTER**: Cloud-free temporal median composite.


In [12]:
from PIL import Image

for i, r in enumerate(results):
    if r["status"] == "SUCCESS":
        vis_path = r["result"]["vis_path"]
        print(f"=== Visualization for Sample {r['sample_id']} ===")
        img = Image.open(vis_path)
        plt.figure(figsize=(14, 5))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"Sample {r['sample_id']} Preprocessing Preview", fontsize=12, fontweight="bold")
        plt.show()


=== Visualization for Sample sample_0000 ===
=== Visualization for Sample sample_0001 ===
=== Visualization for Sample sample_0002 ===
=== Visualization for Sample sample_0003 ===
=== Visualization for Sample sample_0004 ===


C:\Users\advik\AppData\Local\Temp\ipykernel_23092\2082798623.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Output Validation
Verify all generated files on disk:
- Each GeoTIFF opens successfully and contains correct bands (4 bands for composite, 1 band for per-band tifs).
- Dimensions, projection/CRS tags, and file sizes are verified.
- `visualization.png` exists and is non-empty.


In [13]:
print("=" * 80)
print(f"{'Sample ID':<15} {'File Name':<30} {'Bands':<8} {'Engine':<20} {'Status'}")
print("=" * 80)

total_tifs_checked = 0
for r in results:
    if r["status"] == "SUCCESS":
        val = r["validation"]
        for fc in val["files_checked"]:
            if "bands" in fc:
                total_tifs_checked += 1
                b = str(fc["bands"])
                eng = fc["engine"]
                print(f"{r['sample_id']:<15} {fc['file']:<30} {b:<8} {eng:<20} {fc['status']}")

print("=" * 80)
print(f"Total GeoTIFFs validated on disk: {total_tifs_checked}")


Sample ID       File Name                      Bands    Engine               Status
sample_0000     composite_reflectance.tif      4        tifffile (rasterio fallback) VALID
sample_0000     composite_norm.tif             4        tifffile (rasterio fallback) VALID
sample_0000     B02.tif                        1        tifffile (rasterio fallback) VALID
sample_0000     B03.tif                        1        tifffile (rasterio fallback) VALID
sample_0000     B04.tif                        1        tifffile (rasterio fallback) VALID
sample_0000     B08.tif                        1        tifffile (rasterio fallback) VALID
sample_0001     composite_reflectance.tif      4        tifffile (rasterio fallback) VALID
sample_0001     composite_norm.tif             4        tifffile (rasterio fallback) VALID
sample_0001     B02.tif                        1        tifffile (rasterio fallback) VALID
sample_0001     B03.tif                        1        tifffile (rasterio fallback) VALID
sample

## 14. Results
Summary of processed outputs and storage directory layout.


In [14]:
out_dir = config.output_root / config.dataset_name
sample_folders = sorted([d for d in out_dir.iterdir() if d.is_dir()])
print(f"Output Root Directory: {out_dir.resolve()}")
print(f"Number of sample folders created: {len(sample_folders)}")
for sf in sample_folders:
    tifs = list(sf.glob("**/*.tif"))
    has_vis = (sf / "visualization.png").exists()
    print(f"  Folder '{sf.name}': {len(tifs)} GeoTIFFs | visualization.png: {'Present' if has_vis else 'MISSING'}")


Output Root Directory: C:\Users\advik\OneDrive\Desktop\ADVIK FILES\Skills\coding\projects\Ours\SIH\v3\processed_dataset\coimbatore_aoi
Number of sample folders created: 5
  Folder '0000': 6 GeoTIFFs | visualization.png: Present
  Folder '0001': 6 GeoTIFFs | visualization.png: Present
  Folder '0002': 6 GeoTIFFs | visualization.png: Present
  Folder '0003': 6 GeoTIFFs | visualization.png: Present
  Folder '0004': 6 GeoTIFFs | visualization.png: Present


## 15. Limitations & Next Steps

### Current Capabilities
1. **PyTorch Integration**: Seamless streaming via `Dataset` and `DataLoader` abstractions.
2. **Cloud Removal**: Effective SCL-guided invalid pixel masking and temporal median compositing.
3. **Standardized Hierarchy**: Uniform folder layout `processed_dataset/<Dataset>/<sample_index>/*.tif` with per-band subfolders.
4. **Verification**: Robust dual-engine validation (`rasterio` in cloud/Linux, `tifffile` fallback on restrictive environments).

### Next Steps (Deliverable 2)
1. **Multi-Dataset Generalization**: Abstract `BaseDataset(ABC)` and create specialized loaders (`DatasetALoader`, `DatasetBLoader`, etc.).
2. **Uniform Interface**: Re-use the identical `SentinelPreprocessor` across multiple heterogeneously structured input datasets.
3. **Hard Cap**: Retain `MAX_SAMPLES_PER_DATASET = 5` for multi-dataset safety.
